## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
## add your code here
#include <stdio.h>
#include <stdlib.h>
#include <assert.h>

#define MAX_N 1030
#define MAX_OPS 50000

int n;
int specialA, specialB;
int blockSize;

int a[MAX_N];          // 当前排列
int pos[MAX_N];        // pos[x] 表示值 x 当前所在的位置

int answer[MAX_OPS];   // 记录所有操作
int ansCnt = 0;


/* 读取整数 */
int read_int() {
    int x = 0;
    int ch = getchar();

    while (ch < '0' || ch > '9') {
        ch = getchar();
    }

    while (ch >= '0' && ch <= '9') {
        x = x * 10 + ch - '0';
        ch = getchar();
    }

    return x;
}


/* 记录一次操作 */
void add_operation(int op) {
    answer[ansCnt++] = op;
}


/* 重新计算每个值所在的位置 */
void update_position() {
    for (int i = 0; i < n; i++) {
        pos[a[i]] = i;
    }
}


/*
    操作 0：
    交换所有值 specialA 和 specialB
*/
void do_swap_operation() {
    add_operation(0);

    for (int i = 0; i < n; i++) {
        if (a[i] == specialA) {
            a[i] = specialB;
        } else if (a[i] == specialB) {
            a[i] = specialA;
        }
    }

    update_position();
}


/*
    操作 2 x：
    所有数加 x 后对 n 取模
*/
void do_add_operation(int x) {
    x %= n;

    if (x < 0) {
        x += n;
    }

    if (x == 0) {
        return;
    }

    add_operation(x);

    for (int i = 0; i < n; i++) {
        a[i] = (a[i] + x) % n;
    }

    update_position();
}


/*
    操作 1 x：
    所有数异或 x
*/
void do_xor_operation(int x) {
    if (x == 0) {
        return;
    }

    add_operation(-x);

    for (int i = 0; i < n; i++) {
        a[i] ^= x;
    }

    update_position();
}


/*
    计算两个值 x, y 映射之后的位置。
    这个函数是后面构造交换操作的辅助函数。
*/
void get_mapped_pair(int x, int y, int *mx, int *my) {
    int diff = (y - x + n - blockSize + n) % n;

    *mx = 0;
    *my = 0;

    for (int step = n / 2; step >= 2 * blockSize; step >>= 1) {
        if (diff >= step) {
            diff -= step;
            *my += step / 2;
        } else {
            *mx += step / 2;
        }
    }

    *mx += n / 2;
    *mx += x & (blockSize - 1);

    *my += x & (blockSize - 1);
}


/*
    用题目允许的三种操作，交换两个值 x 和 y。
*/
void swap_two_values(int x, int y) {
    int groupX = x / blockSize;
    int groupY = y / blockSize;

    /*
        如果 x 和 y 在同类分组里，
        需要找一个中间值 middle 进行三次交换。
    */
    if (groupX % 2 == groupY % 2) {
        int middle;

        if (groupX % 2 == 0) {
            middle = (x & (blockSize - 1)) + blockSize;
        } else {
            middle = x & (blockSize - 1);
        }

        swap_two_values(x, middle);
        swap_two_values(y, middle);
        swap_two_values(x, middle);

        return;
    }

    int mapA, mapB;
    int mapX, mapY;

    get_mapped_pair(specialA, specialB, &mapA, &mapB);
    get_mapped_pair(x, y, &mapX, &mapY);

    do_add_operation((mapX - x + n) % n);
    do_xor_operation(mapX ^ mapA);
    do_add_operation((specialA - mapA + n) % n);

    do_swap_operation();

    do_add_operation((mapA - specialA + n) % n);
    do_xor_operation(mapX ^ mapA);
    do_add_operation((x - mapX + n) % n);
}


/* 下面这个结构体用于处理缩小后的排列 */
typedef struct {
    int val[MAX_N];
    int len;

    int ops[MAX_OPS];
    int opCnt;
} SmallPermutation;


/* 给 SmallPermutation 添加操作 */
void small_add_op(SmallPermutation *p, int op) {
    p->ops[p->opCnt++] = op;
}


/*
    递归构造缩小排列需要的操作。
    返回 1 表示可以构造，返回 0 表示不可以。
*/
int build_small_permutation(SmallPermutation *p) {
    int used[MAX_N] = {0};

    /*
        检查 p->val[0..len-1] 是否是 0..len-1 的一个排列
    */
    for (int i = 0; i < p->len; i++) {
        if (p->val[i] < 0 || p->val[i] >= p->len) {
            return 0;
        }

        used[p->val[i]] = 1;
    }

    for (int i = 0; i < p->len; i++) {
        if (!used[i]) {
            return 0;
        }
    }

    if (p->len == 1) {
        return 1;
    }

    SmallPermutation leftPart;
    SmallPermutation rightPart;

    leftPart.len = p->len / 2;
    rightPart.len = p->len / 2;
    leftPart.opCnt = 0;
    rightPart.opCnt = 0;

    for (int i = 0; i < p->len / 2; i++) {
        leftPart.val[i] = p->val[i * 2] / 2;
        rightPart.val[i] = p->val[i * 2 + 1] / 2;
    }

    if (!build_small_permutation(&leftPart)) {
        return 0;
    }

    if (!build_small_permutation(&rightPart)) {
        return 0;
    }

    if (p->val[0] & 1) {
        if (p->len == 2) {
            small_add_op(p, 1);
        } else {
            small_add_op(p, -1);
        }
    }

    int leftXor = 0;

    for (int i = 0; i < leftPart.opCnt; i++) {
        int op = leftPart.ops[i];

        if (op > 0) {
            small_add_op(p, -1);
            small_add_op(p, 1);
        } else {
            small_add_op(p, op * 2);
            leftXor ^= (-op) * 2;
        }
    }

    if (leftXor != 0) {
        small_add_op(p, -leftXor);
    }

    int rightXor = 0;

    for (int i = 0; i < rightPart.opCnt; i++) {
        int op = rightPart.ops[i];

        if (op > 0) {
            small_add_op(p, 1);
            small_add_op(p, -1);
        } else {
            small_add_op(p, op * 2);
            rightXor ^= (-op) * 2;
        }
    }

    if ((rightXor & (p->len / 2)) != (leftXor & (p->len / 2))) {
        return 0;
    }

    if (leftXor >= p->len / 2) {
        leftXor -= p->len / 2;
    }

    if (rightXor >= p->len / 2) {
        rightXor -= p->len / 2;
    }

    if (leftXor != rightXor) {
        return 0;
    }

    /*
        合并连续的异或操作。
        因为连续两次异或可以合成一次异或。
    */
    int merged[MAX_OPS];
    int mergedCnt = 0;

    for (int i = 0; i < p->opCnt; i++) {
        int op = p->ops[i];

        if (mergedCnt == 0) {
            merged[mergedCnt++] = op;
        } else {
            if (op < 0 && merged[mergedCnt - 1] < 0) {
                merged[mergedCnt - 1] =
                    -((-merged[mergedCnt - 1]) ^ (-op));

                if (merged[mergedCnt - 1] == 0) {
                    mergedCnt--;
                }
            } else {
                merged[mergedCnt++] = op;
            }
        }
    }

    p->opCnt = mergedCnt;

    for (int i = 0; i < mergedCnt; i++) {
        p->ops[i] = merged[i];
    }

    return 1;
}


/* qsort 的比较函数 */
int cmp_int(const void *p1, const void *p2) {
    int x = *(const int *)p1;
    int y = *(const int *)p2;

    if (x < y) return -1;
    if (x > y) return 1;
    return 0;
}


int main() {
    n = read_int();
    specialA = read_int();
    specialB = read_int();

    for (int i = 0; i < n; i++) {
        a[i] = read_int();
    }

    update_position();

    /*
        blockSize 表示当前操作能力能影响到的最小分组大小。
    */
    blockSize = (specialA - specialB + n) % n;
    blockSize &= -blockSize;

    if (blockSize == 0) {
        blockSize = n;
    }

    /*
        先处理低位上的排列关系。
    */
    if (blockSize > 1) {
        SmallPermutation base;

        base.len = blockSize;
        base.opCnt = 0;

        for (int i = 0; i < blockSize; i++) {
            base.val[i] = a[i] & (blockSize - 1);
        }

        if (!build_small_permutation(&base)) {
            printf("-1\n");
            return 0;
        }

        for (int i = 0; i < base.opCnt; i++) {
            int op = base.ops[i];

            if (op > 0) {
                do_add_operation(op);
            } else {
                do_xor_operation(-op);
            }
        }
    }

    /*
        按照 blockSize 分组，逐组检查并调整。
    */
    for (int r = 0; r < blockSize; r++) {
        int bucket[MAX_N];
        int bucketCnt = 0;

        for (int j = r; j < n; j += blockSize) {
            bucket[bucketCnt++] = a[j];
        }

        qsort(bucket, bucketCnt, sizeof(int), cmp_int);

        int ok = 1;
        int idx = 0;

        for (int j = r; j < n; j += blockSize) {
            if (bucket[idx] != j) {
                ok = 0;
                break;
            }

            idx++;
        }

        if (!ok) {
            printf("-1\n");
            return 0;
        }

        /*
            如果当前位置 j 上的数不是 j，
            就通过构造操作把它换回来。
        */
        for (int j = r; j < n; j += blockSize) {
            if (a[j] != j) {
                swap_two_values(j, a[j]);
            }
        }
    }

    /*
        最后检查是否已经变成有序排列：
        a[i] == i
    */
    for (int i = 0; i < n; i++) {
        assert(a[i] == i);
    }

    printf("%d\n", ansCnt);

    for (int i = 0; i < ansCnt; i++) {
        int op = answer[i];

        if (op == 0) {
            printf("0\n");
        } else if (op < 0) {
            printf("1 %d\n", -op);
        } else {
            printf("2 %d\n", op);
        }
    }

    return 0;
}

## B 长跑

In [ ]:
## add your code here
#include <stdio.h>
#include <stdlib.h>

#define INF 1000000000

typedef struct {
    int pos;
    int cost;
} Station;

int cmp(const void *a, const void *b) {
    Station *x = (Station *)a;
    Station *y = (Station *)b;

    if (x->pos != y->pos) return x->pos - y->pos;
    return x->cost - y->cost;
}

int main() {
    int N, L, Maxn, S;

    while (scanf("%d %d %d %d", &N, &L, &Maxn, &S) == 4) {
        Station st[2005];

        for (int i = 0; i < N; i++) {
            scanf("%d %d", &st[i].pos, &st[i].cost);
        }

        qsort(st, N, sizeof(Station), cmp);

        /*
            合并同一位置的补给站：
            如果同一个位置有多个补给站，只保留花费最少的那个。
        */
        Station node[2005];
        int m = 0;

        node[m].pos = 0;
        node[m].cost = 0;
        m++;

        for (int i = 0; i < N; i++) {
            if (st[i].pos > L) continue;

            if (m > 1 && node[m - 1].pos == st[i].pos) {
                if (st[i].cost < node[m - 1].cost) {
                    node[m - 1].cost = st[i].cost;
                }
            } else {
                node[m].pos = st[i].pos;
                node[m].cost = st[i].cost;
                m++;
            }
        }

        int dp[2005];

        for (int i = 0; i < m; i++) {
            dp[i] = INF;
        }

        dp[0] = 0;

        /*
            dp[i] 表示：
            到达第 i 个补给点，并且在这里补满体力后，最少花费多少硬币。
        */
        for (int i = 1; i < m; i++) {
            for (int j = 0; j < i; j++) {
                if (node[i].pos - node[j].pos <= Maxn) {
                    if (dp[j] + node[i].cost < dp[i]) {
                        dp[i] = dp[j] + node[i].cost;
                    }
                }
            }
        }

        int ok = 0;

        /*
            从起点直接到终点
        */
        if (L <= Maxn) {
            ok = 1;
        }

        /*
            从某个补给点补满后到终点
        */
        for (int i = 1; i < m; i++) {
            if (dp[i] <= S && L - node[i].pos <= Maxn) {
                ok = 1;
                break;
            }
        }

        if (ok) {
            printf("Yes\n");
        } else {
            printf("No\n");
        }
    }

    return 0;
}

## C 最长回文

In [ ]:
## add your code here
#include <stdio.h>
#include <string.h>

#define MAXN 100000

typedef unsigned long long ull;

const ull BASE1 = 911382323ULL;
const ull BASE2 = 972663749ULL;

int n;
char A[MAXN + 5], B[MAXN + 5], RBs[MAXN + 5];

int palA[2 * MAXN + 5], palB[2 * MAXN + 5];
int d1[MAXN + 5], d2[MAXN + 5];
int memo[2 * MAXN + 5];

ull pow1_[MAXN + 5], pow2_[MAXN + 5];
ull hA1[MAXN + 5], hA2[MAXN + 5];
ull hR1[MAXN + 5], hR2[MAXN + 5];

int min_int(int a, int b) {
    return a < b ? a : b;
}

void manacher(char s[], int R[]) {
    int i, k, l, r;

    for (i = 0; i <= 2 * n + 1; i++) {
        R[i] = 0;
    }

    l = 1;
    r = 0;

    for (i = 1; i <= n; i++) {
        if (i > r) {
            k = 1;
        } else {
            k = min_int(d1[l + r - i], r - i + 1);
        }

        while (i - k >= 1 && i + k <= n && s[i - k] == s[i + k]) {
            k++;
        }

        d1[i] = k;
        R[2 * i] = 2 * k - 1;

        if (i + k - 1 > r) {
            l = i - k + 1;
            r = i + k - 1;
        }
    }

    l = 1;
    r = 0;

    for (i = 1; i <= n; i++) {
        if (i > r) {
            k = 0;
        } else {
            k = min_int(d2[l + r - i + 1], r - i + 1);
        }

        while (i - k - 1 >= 1 && i + k <= n && s[i - k - 1] == s[i + k]) {
            k++;
        }

        d2[i] = k;
        R[2 * i - 1] = 2 * k;

        if (i + k - 1 > r) {
            l = i - k;
            r = i + k - 1;
        }
    }
}

void build_hash(char s[], ull h1[], ull h2[]) {
    int i;

    h1[0] = 0;
    h2[0] = 0;

    for (i = 1; i <= n; i++) {
        ull v = (ull)(s[i] - 'A' + 1);
        h1[i] = h1[i - 1] * BASE1 + v;
        h2[i] = h2[i - 1] * BASE2 + v;
    }
}

ull get_hash(ull h[], ull pw[], int l, int len) {
    return h[l + len - 1] - h[l - 1] * pw[len];
}

int same_substring(int pa, int pr, int len) {
    ull a1, b1, a2, b2;

    if (len == 0) return 1;

    a1 = get_hash(hA1, pow1_, pa, len);
    b1 = get_hash(hR1, pow1_, pr, len);

    if (a1 != b1) return 0;

    a2 = get_hash(hA2, pow2_, pa, len);
    b2 = get_hash(hR2, pow2_, pr, len);

    return a2 == b2;
}

int lcp_A_RB(int pa, int pr, int limit) {
    int left = 0;
    int right = limit;

    while (left < right) {
        int mid = (left + right + 1) / 2;

        if (same_substring(pa, pr, mid)) {
            left = mid;
        } else {
            right = mid - 1;
        }
    }

    return left;
}

int check_one_side(int L, int Rpal[], int type) {
    int center;

    for (center = 1; center <= 2 * n + 1; center++) {
        int R = Rpal[center];
        int d, aa, bb;
        int limit, pa, pr, g;

        if ((L & 1) != ((center + 1) & 1)) {
            continue;
        }

        if (R >= L) {
            return 1;
        }

        if (type == 0) {
            d = center - 1;
        } else {
            d = center + 1;
        }

        if ((d - L) % 2 != 0) {
            continue;
        }

        aa = (d - L) / 2;
        bb = (d + L) / 2;

        if (aa < 0 || aa > n || bb < 1 || bb > n + 1) {
            continue;
        }

        limit = L / 2;

        if (n - aa < limit) {
            limit = n - aa;
        }

        if (bb - 1 < limit) {
            limit = bb - 1;
        }

        if (limit <= 0) {
            continue;
        }

        pa = aa + 1;
        pr = n - bb + 2;

        if (pa < 1 || pa > n || pr < 1 || pr > n) {
            continue;
        }

        g = lcp_A_RB(pa, pr, limit);

        if (L - 2 * g <= R) {
            return 1;
        }
    }

    return 0;
}

int check_exact(int L) {
    if (L == 0) return 1;

    if (memo[L] != -1) {
        return memo[L];
    }

    if (check_one_side(L, palA, 0)) {
        memo[L] = 1;
        return 1;
    }

    if (check_one_side(L, palB, 1)) {
        memo[L] = 1;
        return 1;
    }

    memo[L] = 0;
    return 0;
}

int check_at_least(int L) {
    if (check_exact(L)) return 1;

    if (L + 1 <= 2 * n && check_exact(L + 1)) {
        return 1;
    }

    return 0;
}

int main() {
    int i;
    int left, right;

    scanf("%d", &n);
    scanf("%s", A + 1);
    scanf("%s", B + 1);

    for (i = 1; i <= n; i++) {
        RBs[i] = B[n - i + 1];
    }

    RBs[n + 1] = '\0';

    pow1_[0] = 1;
    pow2_[0] = 1;

    for (i = 1; i <= n; i++) {
        pow1_[i] = pow1_[i - 1] * BASE1;
        pow2_[i] = pow2_[i - 1] * BASE2;
    }

    build_hash(A, hA1, hA2);
    build_hash(RBs, hR1, hR2);

    manacher(A, palA);
    manacher(B, palB);

    for (i = 0; i <= 2 * n + 1; i++) {
        memo[i] = -1;
    }

    left = 0;
    right = 2 * n;

    while (left < right) {
        int mid = (left + right + 1) / 2;

        if (check_at_least(mid)) {
            left = mid;
        } else {
            right = mid - 1;
        }
    }

    printf("%d\n", left);

    return 0;
}

## D 优惠券

In [ ]:
## add your code here
#include <stdio.h>
#include <string.h>

#define MAXM 500005
#define MAXX 100005

int bit[MAXM];
int lastPos[MAXX];
char lastOp[MAXX];

int n;

void add(int idx, int val) {
    while (idx <= n) {
        bit[idx] += val;
        idx += idx & -idx;
    }
}

int sum(int idx) {
    int res = 0;
    while (idx > 0) {
        res += bit[idx];
        idx -= idx & -idx;
    }
    return res;
}

// 找到第 k 个还没被使用的 ?
int kth(int k) {
    int idx = 0;
    int step = 1;

    while ((step << 1) <= n) {
        step <<= 1;
    }

    while (step > 0) {
        int next = idx + step;
        if (next <= n && bit[next] < k) {
            idx = next;
            k -= bit[next];
        }
        step >>= 1;
    }

    return idx + 1;
}

// 需要在 [L, R] 之间找一个没用过的 ?
int useQuestion(int L, int R) {
    if (L > R) return 0;

    int before = sum(L - 1);
    int total = sum(R);

    if (total - before <= 0) {
        return 0;
    }

    int pos = kth(before + 1);
    add(pos, -1);

    return 1;
}

int main() {
    int m;

    while (scanf("%d", &m) != EOF) {
        n = m;

        memset(bit, 0, sizeof(int) * (m + 2));
        memset(lastOp, 0, sizeof(lastOp));
        memset(lastPos, 0, sizeof(lastPos));

        int ans = -1;

        for (int i = 1; i <= m; i++) {
            char op[10];
            scanf("%s", op);

            if (op[0] == 'I' || op[0] == 'O') {
                int x;
                scanf("%d", &x);

                if (ans != -1) {
                    continue;
                }

                if (lastOp[x] == 0) {
                    // 第一次出现就是 O x，前面必须有 ? 补成 I x
                    if (op[0] == 'O') {
                        if (!useQuestion(1, i - 1)) {
                            ans = i;
                        }
                    }
                } else {
                    // 两次相同操作，中间必须有 ? 补一个相反操作
                    if (lastOp[x] == op[0]) {
                        if (!useQuestion(lastPos[x] + 1, i - 1)) {
                            ans = i;
                        }
                    }
                }

                lastOp[x] = op[0];
                lastPos[x] = i;
            } else {
                // ? 或中文 ？
                if (ans == -1) {
                    add(i, 1);
                }
            }
        }

        printf("%d\n", ans);
    }

    return 0;
}

## E 任意点

In [ ]:
## add your code here
#include <stdio.h>

#define MAXN 105

int fa[MAXN];
int x[MAXN], y[MAXN];

int find(int a) {
    if (fa[a] != a) {
        fa[a] = find(fa[a]);
    }
    return fa[a];
}

void unite(int a, int b) {
    int ra = find(a);
    int rb = find(b);
    if (ra != rb) {
        fa[ra] = rb;
    }
}

int main() {
    int n;
    scanf("%d", &n);

    for (int i = 1; i <= n; i++) {
        scanf("%d %d", &x[i], &y[i]);
        fa[i] = i;
    }

    for (int i = 1; i <= n; i++) {
        for (int j = i + 1; j <= n; j++) {
            if (x[i] == x[j] || y[i] == y[j]) {
                unite(i, j);
            }
        }
    }

    int cnt = 0;
    for (int i = 1; i <= n; i++) {
        if (find(i) == i) {
            cnt++;
        }
    }

    printf("%d\n", cnt - 1);

    return 0;
}

## F 通配符匹配

In [ ]:
## add your code here
#include <stdio.h>
#include <string.h>

#define MAXL 100000
#define MAXTOK 1005

char P[MAXL + 5];
char S[MAXL + 5];

int piPool[MAXL + 5];
int markArr[MAXL + 5];
int seenArr[MAXL + 5];
int curStamp = 0;

typedef struct {
    int start;      // 该普通字母块在模式串 P 中的位置
    int len;        // 普通字母块长度
    int offset;     // 该普通字母块在当前段中的偏移
    int piStart;    // KMP 的 pi 数组在 piPool 中的位置
} Part;

typedef struct {
    int len;        // 当前段总长度，包含 ? 和普通字母
    int firstPart;  // 当前段第一个普通字母块编号
    int partCnt;    // 当前段中普通字母块数量
} Segment;

Part parts[MAXTOK];
Segment segs[MAXTOK];

int partCnt = 0;
int segCnt = 0;
int m;
int hasStar;
int leadingStar;
int trailingStar;

void build_pi_for_part(int id) {
    int len = parts[id].len;
    int st = parts[id].start;
    int ps = parts[id].piStart;

    piPool[ps] = 0;

    for (int i = 1; i < len; i++) {
        int j = piPool[ps + i - 1];

        while (j > 0 && P[st + i] != P[st + j]) {
            j = piPool[ps + j - 1];
        }

        if (P[st + i] == P[st + j]) {
            j++;
        }

        piPool[ps + i] = j;
    }
}

void parse_pattern() {
    m = strlen(P);

    hasStar = 0;
    leadingStar = (P[0] == '*');
    trailingStar = (P[m - 1] == '*');

    for (int i = 0; i < m; i++) {
        if (P[i] == '*') {
            hasStar = 1;
        }
    }

    partCnt = 0;
    segCnt = 0;

    int piUsed = 0;
    int i = 0;

    while (i < m) {
        if (P[i] == '*') {
            i++;
            continue;
        }

        int l = i;

        while (i < m && P[i] != '*') {
            i++;
        }

        int r = i;

        segs[segCnt].len = r - l;
        segs[segCnt].firstPart = partCnt;
        segs[segCnt].partCnt = 0;

        int j = l;

        while (j < r) {
            if (P[j] == '?') {
                j++;
            } else {
                int st = j;

                while (j < r && P[j] != '?') {
                    j++;
                }

                parts[partCnt].start = st;
                parts[partCnt].len = j - st;
                parts[partCnt].offset = st - l;
                parts[partCnt].piStart = piUsed;

                piUsed += parts[partCnt].len;

                build_pi_for_part(partCnt);

                partCnt++;
                segs[segCnt].partCnt++;
            }
        }

        segCnt++;
    }
}

int segment_match_at(int sid, int start, char text[], int n) {
    Segment *sg = &segs[sid];

    if (start < 0) return 0;
    if (start + sg->len > n) return 0;

    for (int i = 0; i < sg->partCnt; i++) {
        Part *pt = &parts[sg->firstPart + i];

        int textPos = start + pt->offset;
        int patPos = pt->start;

        if (memcmp(text + textPos, P + patPos, pt->len) != 0) {
            return 0;
        }
    }

    return 1;
}

void mark_occurrences_of_part(int pid, char text[], int n, int minStart, int maxStart) {
    Part *pt = &parts[pid];

    int q = 0;
    int ps = pt->piStart;

    for (int i = 0; i < n; i++) {
        while (q > 0 && text[i] != P[pt->start + q]) {
            q = piPool[ps + q - 1];
        }

        if (text[i] == P[pt->start + q]) {
            q++;
        }

        if (q == pt->len) {
            int occStart = i - pt->len + 1;
            int candStart = occStart - pt->offset;

            if (candStart >= minStart && candStart <= maxStart) {
                if (seenArr[candStart] != curStamp) {
                    seenArr[candStart] = curStamp;
                    markArr[candStart] = 0;
                }

                markArr[candStart]++;
            }

            q = piPool[ps + q - 1];
        }
    }
}

int find_segment(int sid, char text[], int n, int minStart, int maxStart) {
    Segment *sg = &segs[sid];

    if (maxStart > n - sg->len) {
        maxStart = n - sg->len;
    }

    if (minStart < 0) {
        minStart = 0;
    }

    if (minStart > maxStart) {
        return -1;
    }

    if (sg->partCnt == 0) {
        return minStart;
    }

    curStamp++;

    for (int i = 0; i < sg->partCnt; i++) {
        int pid = sg->firstPart + i;
        mark_occurrences_of_part(pid, text, n, minStart, maxStart);
    }

    for (int st = minStart; st <= maxStart; st++) {
        if (seenArr[st] == curStamp && markArr[st] == sg->partCnt) {
            return st;
        }
    }

    return -1;
}

int match_file(char text[]) {
    int n = strlen(text);

    /*
        如果模式串里没有 *，
        那么文件名长度必须和模式串长度完全相等。
    */
    if (!hasStar) {
        if (m != n) return 0;
        return segment_match_at(0, 0, text, n);
    }

    /*
        模式串全是 *，
        可以匹配任意文件名。
    */
    if (segCnt == 0) {
        return 1;
    }

    int pos = 0;
    int idx = 0;

    /*
        如果模式串不是以 * 开头，
        那么第一段必须贴着文件名开头匹配。
    */
    if (!leadingStar) {
        if (!segment_match_at(0, 0, text, n)) {
            return 0;
        }

        pos = segs[0].len;
        idx = 1;
    }

    int lastAnchor = -1;
    int endStart = -1;

    /*
        如果模式串不是以 * 结尾，
        那么最后一段必须贴着文件名结尾匹配。
    */
    if (!trailingStar) {
        lastAnchor = segCnt - 1;

        if (lastAnchor >= idx) {
            endStart = n - segs[lastAnchor].len;

            if (endStart < 0) return 0;

            if (!segment_match_at(lastAnchor, endStart, text, n)) {
                return 0;
            }
        }
    }

    int middleEnd;

    if (lastAnchor >= idx) {
        middleEnd = lastAnchor;
    } else {
        middleEnd = segCnt;
    }

    /*
        中间的段只需要按顺序出现即可。
    */
    for (int k = idx; k < middleEnd; k++) {
        int maxStart;

        if (lastAnchor >= idx) {
            maxStart = endStart - segs[k].len;
        } else {
            maxStart = n - segs[k].len;
        }

        int st = find_segment(k, text, n, pos, maxStart);

        if (st == -1) {
            return 0;
        }

        pos = st + segs[k].len;
    }

    /*
        如果最后一段被固定在结尾，
        前面的内容不能和它重叠。
    */
    if (lastAnchor >= idx) {
        if (pos > endStart) return 0;
    }

    return 1;
}

int main() {
    int n;

    scanf("%s", P);
    scanf("%d", &n);

    parse_pattern();

    while (n--) {
        scanf("%s", S);

        if (match_file(S)) {
            printf("YES\n");
        } else {
            printf("NO\n");
        }
    }

    return 0;
}

## G 汉诺塔

In [ ]:
## add your code here
#include <stdio.h>

typedef long long ll;

int id(char c) {
    if (c == 'A') return 0;
    if (c == 'B') return 1;
    return 2;
}

int main() {
    int n;
    char op[10];

    int rank[3][3];

    scanf("%d", &n);

    for (int i = 0; i < 6; i++) {
        scanf("%s", op);
        int x = id(op[0]);
        int y = id(op[1]);
        rank[x][y] = i;
    }

    /*
        to[k][s]：
        k 个盘子一开始都在 s 柱，
        按策略操作后，它们第一次整体移动到哪根柱子。

        step[k][s]：
        对应需要多少步。
    */
    int to[35][3];
    ll step[35][3];

    /*
        只有 1 个盘子时，直接看从当前柱子出发的两个操作谁优先级高。
    */
    for (int s = 0; s < 3; s++) {
        int a = (s + 1) % 3;
        int b = (s + 2) % 3;

        if (rank[s][a] < rank[s][b]) {
            to[1][s] = a;
        } else {
            to[1][s] = b;
        }

        step[1][s] = 1;
    }

    for (int k = 2; k <= n; k++) {
        for (int start = 0; start < 3; start++) {
            ll ans = 0;

            /*
                最大盘子 k 一开始在 start。
                先让 k-1 个小盘子从 start 移走。
            */
            int smallPos = start;

            int nextSmall = to[k - 1][smallPos];
            ans += step[k - 1][smallPos];

            /*
                小盘子去了 nextSmall，
                最大盘子只能移动到第三根柱子。
            */
            int bigPos = 3 - start - nextSmall;
            ans++;

            smallPos = nextSmall;

            /*
                之后反复移动 k-1 个小盘子。
                如果小盘子移动到了 bigPos，
                那么 k 个盘子就整体完成了。
                否则最大盘子继续移动。
            */
            while (1) {
                nextSmall = to[k - 1][smallPos];
                ans += step[k - 1][smallPos];

                if (nextSmall == bigPos) {
                    to[k][start] = bigPos;
                    step[k][start] = ans;
                    break;
                }

                /*
                    小盘子没到最大盘子所在柱，
                    最大盘子继续移动到剩下的第三根柱子。
                */
                bigPos = 3 - bigPos - nextSmall;
                ans++;

                smallPos = nextSmall;
            }
        }
    }

    /*
        初始所有盘子在 A，也就是编号 0。
    */
    printf("%lld\n", step[n][0]);

    return 0;
}

## H 马步距离

In [ ]:
## add your code here
#include <stdio.h>
#include <stdlib.h>

typedef long long ll;

ll max_ll(ll a, ll b) {
    return a > b ? a : b;
}

int main() {
    ll xp, yp, xs, ys;
    scanf("%lld %lld %lld %lld", &xp, &yp, &xs, &ys);

    ll dx = llabs(xp - xs);
    ll dy = llabs(yp - ys);

    if (dx < dy) {
        ll t = dx;
        dx = dy;
        dy = t;
    }

    /*
        特判：
        从 (0,0) 到 (1,0) 至少要 3 步
    */
    if (dx == 1 && dy == 0) {
        printf("3\n");
        return 0;
    }

    /*
        特判：
        从 (0,0) 到 (2,2) 至少要 4 步
    */
    if (dx == 2 && dy == 2) {
        printf("4\n");
        return 0;
    }

    /*
        一般公式：
        1. 每步横向最多接近 2，所以至少 (dx + 1) / 2 步
        2. 每步横纵总距离最多接近 3，所以至少 (dx + dy + 2) / 3 步
    */
    ll ans = max_ll((dx + 1) / 2, (dx + dy + 2) / 3);

    /*
        马每走一步，x + y 的奇偶性会改变。
        所以步数 ans 的奇偶性必须和 dx + dy 一致。
    */
    if ((ans + dx + dy) % 2 != 0) {
        ans++;
    }

    printf("%lld\n", ans);

    return 0;
}

## I 直方图最大矩形

In [ ]:
## add your code here
/**
 * 代码中的类名、方法名、参数名已经指定，请勿修改，直接返回方法规定的值即可
 *
 * 
 * @param heights int整型一维数组 
 * @param heightsLen int heights数组长度
 * @return int整型
 *
 * C语言声明定义全局变量请加上static，防止重复定义
 */
int largestRectangleArea(int* heights, int heightsLen ) {
    int st[100005];

    int top = 0;
    int ans = 0;

    st[0] = -1;

    for (int i = 0; i <= heightsLen; i++) {
        int cur;

        if (i == heightsLen) {
            cur = 0;
        } else {
            cur = heights[i];
        }

        while (top > 0 && heights[st[top]] > cur) {
            int h = heights[st[top]];
            top--;

            int width = i - st[top] - 1;
            int area = h * width;

            if (area > ans) {
                ans = area;
            }
        }

        st[++top] = i;
    }

    return ans;
}

## J 消防局的设立

In [ ]:
## add your code here
#include <stdio.h>
#include <stdlib.h>

#define INF 1000000000

int main() {
    int n;
    scanf("%d", &n);

    int *fa = (int *)calloc(n + 1, sizeof(int));
    int *depth = (int *)calloc(n + 1, sizeof(int));

    int *headDepth = (int *)calloc(n + 1, sizeof(int));
    int *nextSameDepth = (int *)calloc(n + 1, sizeof(int));

    int *down = (int *)malloc((n + 1) * sizeof(int));
    int *childStation = (int *)calloc(n + 1, sizeof(int));
    char *station = (char *)calloc(n + 1, sizeof(char));

    for (int i = 1; i <= n; i++) {
        down[i] = INF;
    }

    int maxDepth = 0;

    headDepth[0] = 1;

    for (int i = 2; i <= n; i++) {
        scanf("%d", &fa[i]);

        depth[i] = depth[fa[i]] + 1;

        if (depth[i] > maxDepth) {
            maxDepth = depth[i];
        }

        nextSameDepth[i] = headDepth[depth[i]];
        headDepth[depth[i]] = i;
    }

    int ans = 0;

    for (int d = maxDepth; d >= 0; d--) {
        for (int v = headDepth[d]; v != 0; v = nextSameDepth[v]) {
            int covered = 0;

            int p = fa[v];
            int gp = 0;

            if (p != 0) {
                gp = fa[p];
            }

            /*
                判断 v 是否已经被覆盖：

                1. v 的子树里 2 步以内有消防局
                2. v 的父亲有消防局
                3. v 的爷爷有消防局
                4. v 的兄弟节点有消防局
            */
            if (down[v] <= 2) {
                covered = 1;
            } else if (p != 0 && station[p]) {
                covered = 1;
            } else if (gp != 0 && station[gp]) {
                covered = 1;
            } else if (p != 0 && childStation[p] > 0) {
                covered = 1;
            }

            if (!covered) {
                /*
                    v 没被覆盖，就把消防局建在 v 的爷爷节点。
                    如果没有爷爷，就尽量往上走。
                */
                int c = v;

                if (fa[c] != 0) {
                    c = fa[c];
                }

                if (fa[c] != 0) {
                    c = fa[c];
                }

                if (!station[c]) {
                    station[c] = 1;
                    ans++;

                    down[c] = 0;

                    int pc = fa[c];

                    if (pc != 0) {
                        childStation[pc]++;

                        if (down[pc] > 1) {
                            down[pc] = 1;
                        }

                        int gpc = fa[pc];

                        if (gpc != 0 && down[gpc] > 2) {
                            down[gpc] = 2;
                        }
                    }
                }
            }
        }
    }

    printf("%d\n", ans);

    free(fa);
    free(depth);
    free(headDepth);
    free(nextSameDepth);
    free(down);
    free(childStation);
    free(station);

    return 0;
}